# Pocket-Agent Colab Runner
Run cells top-to-bottom on Colab T4. This notebook prints a clear submission readiness summary.

In [ ]:
REPO_URL = "https://github.com/ahmedbilal008/vyro-hackathon-submission.git"
!git clone $REPO_URL
%cd vyro-hackathon-submission
!pip -q install --upgrade pip setuptools wheel
!pip -q install --prefer-binary -r requirements.txt
!python starter/build_starter_files.py --manual data/manual_examples.jsonl --out starter
print("[PASS] Setup complete: repo cloned, dependencies installed, starter pack ready")

In [ ]:
import os
cmd = "python data/generate_data.py --out data/train.jsonl --n 2000 --manual data/manual_examples.jsonl"
if os.path.exists("starter/public_test.jsonl"):
    cmd += " --avoid starter/public_test.jsonl"
if os.path.exists("starter/teacher_examples.jsonl"):
    cmd += " --teacher starter/teacher_examples.jsonl"
print("Running:", cmd)
!$cmd
if os.path.exists("starter/public_test.jsonl"):
    !python data/check_overlap.py --train data/train.jsonl --public starter/public_test.jsonl
    print("[PASS] Zero-overlap check completed")
else:
    print("[INFO] starter/public_test.jsonl not found, overlap check skipped")
print("[PASS] Data generation complete")

In [ ]:
!python train.py --model_id Qwen/Qwen2.5-0.5B-Instruct --data data/train.jsonl --out models/adapter
print("[PASS] Adapter training complete at models/adapter")

In [ ]:
import os
import shlex
import subprocess

REPO_DIR = "/content/vyro-hackathon-submission"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

if not os.path.exists("train.py"):
    raise FileNotFoundError(
        f"train.py not found in {os.getcwd()}. "
        "Run setup cell first and make sure you are inside /content/vyro-hackathon-submission."
    )
if not os.path.isdir("models/adapter"):
    raise FileNotFoundError("models/adapter not found. Run the training cell first.")

os.makedirs("models/quantized", exist_ok=True)

def run_quant(mode, extra_args=None):
    cmd = [
        "python", "quantize.py",
        "--base", "Qwen/Qwen2.5-0.5B-Instruct",
        "--adapter", "models/adapter",
        "--out", "models/quantized/model.gguf",
        "--quant", mode,
    ] + (extra_args or [])
    print("[RUN]", " ".join(shlex.quote(x) for x in cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print(f"[ERROR] {mode} failed with code {proc.returncode}")
        if proc.stderr:
            print(proc.stderr[-8000:])
    return proc.returncode

quant_mode = None
if run_quant("q3_k_s", ["--max_mb", "250"]) == 0:
    quant_mode = "q3_k_s"
else:
    print("[INFO] q3_k_s failed or exceeded 250MB, trying q4_k_m")
    if run_quant("q4_k_m") == 0:
        quant_mode = "q4_k_m"
    else:
        print("[INFO] q4_k_m failed, trying q4_0")
        if run_quant("q4_0") == 0:
            quant_mode = "q4_0"

if quant_mode is None or not os.path.exists("models/quantized/model.gguf"):
    raise RuntimeError("Quantization failed for q3_k_s, q4_k_m, and q4_0. Check logs above.")

size_mb = os.path.getsize("models/quantized/model.gguf") / (1024 * 1024)
print(f"[PASS] Quantization complete using {quant_mode}; size={size_mb:.2f} MB")
print(f"[CHECK] <=500MB gate: {'PASS' if size_mb <= 500 else 'FAIL'}")
print(f"[CHECK] <=250MB bonus: {'PASS' if size_mb <= 250 else 'MISS'}")

In [ ]:
from inference import run
out1 = run("weather in Lahore in C", [])
out2 = run("convert that to euros", [])
print("Smoke 1:", out1)
print("Smoke 2:", out2)
print("[PASS] inference.py run() callable and producing outputs")
!python eval_public.py --test starter/public_test.jsonl --out eval_public_summary.json

In [ ]:
print("Commit adapter from Colab only after training is done:")
print("!git config user.name \"Ahmed Bilal\"")
print("!git config user.email \"your-email@example.com\"")
print("!git add models/adapter")
print("!git commit -m \"add trained adapter\"")
print("import getpass")
print("token = getpass.getpass('GitHub token: ')")
print("!git remote set-url origin https://{token}@github.com/ahmedbilal008/vyro-hackathon-submission.git")
print("!git push origin main")

In [ ]:
!zip -r adapter_only.zip models/adapter
print("[PASS] Created adapter_only.zip")
try:
    from google.colab import files
    files.download("adapter_only.zip")
except Exception as e:
    print("[INFO] Auto-download not available in this environment:", e)

In [ ]:
import os
print("========== SUBMISSION READINESS =========='")
print("1) Adapter exists: ", os.path.exists("models/adapter"))
print("2) Quantized model exists: ", os.path.exists("models/quantized/model.gguf"))
print("3) Grader entry exists: ", os.path.exists("inference.py"))
print("4) Demo exists: ", os.path.exists("demo/app.py"))
print("5) Starter public test exists: ", os.path.exists("starter/public_test.jsonl"))
print("6) Eval summary exists: ", os.path.exists("eval_public_summary.json"))
print("Submit this URL: https://github.com/ahmedbilal008/vyro-hackathon-submission")

In [ ]:
import os
os.environ["MODEL_PATH"] = "models/quantized/model.gguf"
print("Launching demo...")
!python demo/app.py